# 04 — BAR-ATH (only if H1 passed; aggregates only)

If H1 failed, nothing is read or written for BAR-ATH. If it passed: 3 full-data `exp4_full` fits, exp_3's fusion rule (tactical input if rejected), two candidate files (decoder selected on provider labels / on `L_ctrl`), official validator, byte-identical rebuild from 3 independent fits, label-free plausibility. No public-window check.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp4_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_4')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp4_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp4_protocol as P
assert C.public_check_spent(), 'exp_1 public check marker missing'

PREREG verified 254e2e216cb05154 | frozen 2026-11-09T17:09:15+00:00


In [2]:
decision = C1.read_json(C.RESULTS / 'decision.json')
if not decision['H1']['passed']:
    summary = {'H1_passed': False, 'files_written': [], 'bar_ath_read': False,
               'statement': decision['adoption']['statement']}
    C1.write_json(C.RESULTS / 'target_none.json', summary)
    print(json.dumps(summary, indent=1))
else:
    import exp4_target as E
    t0 = time.time()
    match, item, fusion = E.inputs()
    models = {arm: E.full_model(arm) for arm in E.MEMBERS}
    summary = E.build_all(match, item, models)
    summary['fusion'] = {k: v for k, v in fusion.items() if k != 'halves'}
    summary['reproduction'] = E.reproduce(match, item)
    summary['exp3_files'] = E.exp3_files()
    C1.write_json(C.RESULTS / 'target_candidates.json', summary)
    keys = ('file', 'sha256', 'dead_share', 'possessions_per_min', 'mean_possession_s', 'possession_count', 'decoder')
    for ls in E.LABEL_SETS:
        print(ls, json.dumps({k: summary[ls].get(k) for k in keys}))
    print('reproduction byte-identical:', summary['reproduction']['byte_identical'], f'| {time.time() - t0:.0f}s')
print('exp_4 proxy fits cached:', P.fit_count(), '| + 1 reproduction check')

{
 "H1_passed": false,
 "files_written": [],
 "bar_ath_read": false,
 "statement": "H1 failed: no exp_4 files are written; exp_3's two BAR-ATH files stay the team's options"
}
exp_4 proxy fits cached: 252 | + 1 reproduction check
